# Aula 05 - Notebook: Formas Normais (FND/FNC) e Otimização de Expressões de Segurança

Neste notebook construímos funções para extrair a Forma Normal Disjuntiva (FND / Mintermos) e a Forma Normal Conjuntiva (FNC / Cláusulas) de funções booleanas da planta de envasamento, comparando a complexidade computacional e o tempo de execução antes e após a simplificação.

In [ ]:
import itertools
import time
from typing import List, Callable, Dict
import pandas as pd

def extrair_formas_normais(variaveis: List[str], funcao: Callable[[Dict[str, bool]], bool]):
    mintermos = []
    maxtermos = []
    
    for combo in itertools.product([False, True], repeat=len(variaveis)):
        estado = dict(zip(variaveis, combo))
        resultado = funcao(estado)
        
        if resultado:
            # Mintermo para FND
            termo = [f"{v}" if estado[v] else f"¬{v}" for v in variaveis]
            mintermos.append("(" + " ∧ ".join(termo) + ")")
        else:
            # Maxtermo para FNC
            termo = [f"¬{v}" if estado[v] else f"{v}" for v in variaveis]
            maxtermos.append("(" + " ∨ ".join(termo) + ")")
            
    fnd_str = " ∨ ".join(mintermos) if mintermos else "FALSO"
    fnc_str = " ∧ ".join(maxtermos) if maxtermos else "VERDADEIRO"
    
    return {
        "Total_Mintermos": len(mintermos),
        "Total_Maxtermos": len(maxtermos),
        "FND": fnd_str,
        "FNC": fnc_str
    }

print("Motor de formas normais carregado.")

## Exemplo: Permissivo da Prensa de Selagem (Setor 400)

In [ ]:
vars_prensa = ['s4', 't1', 'e1', 'byp']

# Expressão Não Otimizada
def prensa_nao_otimizada(st: Dict[str, bool]) -> bool:
    s4, t1, e1, byp = st['s4'], st['t1'], st['e1'], st['byp']
    t_normal = s4 and t1 and (not e1)
    t_manutencao = s4 and t1 and e1 and byp
    t_redundante = s4 and (not s4) and t1 # termo redundante
    return t_normal or t_manutencao or t_redundante

# Expressão Otimizada por Álgebra Booleana
def prensa_otimizada(st: Dict[str, bool]) -> bool:
    return st['s4'] and st['t1'] and ((not st['e1']) or st['byp'])

# Análise Canônica
analise = extrair_formas_normais(vars_prensa, prensa_nao_otimizada)
print(f"Total de Mintermos (FND): {analise['Total_Mintermos']}")
print(f"Total de Maxtermos (FNC): {analise['Total_Maxtermos']}")
print("\nFND Canônica:")
print(analise['FND'])
print("\nFNC Canônica:")
print(analise['FNC'])


## Benchmark de Desempenho e Validação de Equivalência Lógica

In [ ]:
# Validação de que ambas produzem o mesmo valor em todos os 2^4 estados
equivalentes = True
for combo in itertools.product([False, True], repeat=len(vars_prensa)):
    st = dict(zip(vars_prensa, combo))
    if prensa_nao_otimizada(st) != prensa_otimizada(st):
        equivalentes = False
        break

print(f"As funções são logicamente equivalentes em 100% dos estados? {equivalentes}")

# Teste de latência com 500.000 iterações de scan
N = 500000
estado_teste = {'s4': True, 't1': True, 'e1': False, 'byp': False}

t0 = time.time()
for _ in range(N):
    _ = prensa_nao_otimizada(estado_teste)
t_nao_otimizado = time.time() - t0

t0 = time.time()
for _ in range(N):
    _ = prensa_otimizada(estado_teste)
t_otimizado = time.time() - t0

df_perf = pd.DataFrame([
    {"Implementação": "Não Otimizada (SOP Bruto)", "Tempo (s)": f"{t_nao_otimizado:.4f}", "Speedup": "1.00x"},
    {"Implementação": "Otimizada (Álgebra Booleana)", "Tempo (s)": f"{t_otimizado:.4f}", "Speedup": f"{t_nao_otimizado/t_otimizado:.2f}x"}
])
df_perf